In [ ]:
# SYSTEMATIC REVIEW - SEMANTIC SCREENING VIA BERT (PROKNOW-C)

# Autor: Matheus Guarato de Almeida
# Instituição: Universidade Federal de Uberlândia (UFU)
# Descrição: Algoritmo de inteligência artificial para triagem semântica e metodológica utilizando a arquitetura BERT.

# 1. AMBIENTAÇÃO E CONFIGURAÇÃO DE DEPENDÊNCIAS
print("Instalando e importando bibliotecas necessárias no ambiente do Colab...")
!pip install -q sentence-transformers pandas openpyxl tqdm

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from tqdm import tqdm

print("\nCarregando o modelo BERT de alta precisão na GPU/Memória...")
# Arquitetura multilíngue otimizada para computar conceitos abstratos
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo carregado com sucesso e pronto para uso.")

# 2. ETAPA 1: TRIAGEM SEMÂNTICA DE TÍTULOS
print("\n--- INICIANDO ETAPA 1: FILTRAGEM DE TÍTULOS ---")
print("Faça o upload do seu arquivo Excel BRUTO (.xlsx) da WoS:")
uploaded_t = files.upload()
file_name_t = list(uploaded_t.keys())[0]

df_titulos = pd.read_excel(file_name_t)
COLUNA_TITULO = 'Title'

if 'Decisao' not in df_titulos.columns: df_titulos['Decisao'] = None
if 'Justificativa' not in df_titulos.columns: df_titulos['Justificativa'] = None

# conceitos âncoras para o modelo calcular aderência semântica
conceito_manter_t = "Innovation, financial performance, firm market value, R&D profitability, business growth, patentes e retorno financeiro."
conceito_eliminar_t = "Human resources, workplace psychology, team motivation, pure engineering materials, education, public policy, consumer behavior."

# gerar os embeddings dos conceitos
embedding_manter_t = model.encode(conceito_manter_t, convert_to_tensor=True)
embedding_eliminar_t = model.encode(conceito_eliminar_t, convert_to_tensor=True)

print("\nProcessando a classificação semântica local dos títulos...")
for idx, row in tqdm(df_titulos.iterrows(), total=len(df_titulos)):
    titulo = str(row[COLUNA_TITULO])

    if not titulo.strip() or titulo.lower() == 'nan':
        df_titulos.at[idx, 'Decisao'] = "ELIMINAR"
        df_titulos.at[idx, 'Justificativa'] = "Título vazio"
        continue

    # Calcula a similaridade do título com os dois blocos de eixos
    embedding_titulo = model.encode(titulo, convert_to_tensor=True)
    sim_manter_t = util.cos_sim(embedding_titulo, embedding_manter_t).item()
    sim_eliminar_t = util.cos_sim(embedding_titulo, embedding_eliminar_t).item()

    # Classificação com a margem de corte calibrada em >= 0.50 para títulos
    if sim_manter_t > sim_eliminar_t and sim_manter_t >= 0.50:
        df_titulos.at[idx, 'Decisao'] = "MANTER"
        df_titulos.at[idx, 'Justificativa'] = f"Alinhamento semântico financeiro positivo ({sim_manter_t:.2f})"
    else:
        df_titulos.at[idx, 'Decisao'] = "ELIMINAR"
        df_titulos.at[idx, 'Justificativa'] = f"Maior aderência a escopos operacionais/psicológicos ou outras áreas ({sim_eliminar_t:.2f})"

output_file_t = "portfolio_classificado_titulos.xlsx"
df_titulos.to_excel(output_file_t, index=False)
print(f"\nEtapa 1 Concluída! Gerando arquivo intermediário: {output_file_t}")
files.download(output_file_t)

# 3. ETAPA 2: TRIAGEM METODOLÓGICA DE RESUMOS
print("\n--- INICIANDO ETAPA 2: FILTRAGEM DE RESUMOS ---")
print("Faça o upload da planilha contendo os artigos pós-teste de citações:")
uploaded_r = files.upload()
file_name_r = list(uploaded_r.keys())[0]

df_resumos = pd.read_excel(file_name_r)
COLUNA_RESUMO = 'Abstract'

# Criando as colunas específicas para o resultado e métricas do filtro de resumos
df_resumos['Decisao_Resumo'] = None
df_resumos['Score_Resumo'] = None

# Calibração fina com critérios restritos focado em dados duros ao nível da firma e econometria
conceito_manter_resumo = (
    "Quantitative empirical study, econometric analysis, regression model, panel data, OLS, "
    "firm performance, accounting metrics, ROA, ROE, Tobin's Q, financial data, corporate value, "
    "R&D spending impact on profitability."
)

# Critérios para rejeição de desvios operacionais, de RH ou abordagens puramente teóricas
conceito_eliminar_resumo = (
    "Qualitative research, literature review, conceptual framework, theoretical paper, "
    "case study n=1, interview analysis, human resources, employee motivation, team psychology, "
    "consumer behavior, marketing strategy, public policy, macroeconomics GDP."
)

# Gerando os embeddings dos critérios restritos
embedding_manter_r = model.encode(conceito_manter_resumo, convert_to_tensor=True)
embedding_eliminar_r = model.encode(conceito_eliminar_resumo, convert_to_tensor=True)

print("\nProcessando a filtragem semântica e metodológica dos resumos...")
for idx, row in tqdm(df_resumos.iterrows(), total=len(df_resumos)):
    resumo = str(row[COLUNA_RESUMO])

    if not resumo.strip() or resumo.lower() == 'nan':
        df_resumos.at[idx, 'Decisao_Resumo'] = "ELIMINAR"
        df_resumos.at[idx, 'Score_Resumo'] = 0.0
        continue

    # Calcula a proximidade do resumo com o perfil metodológico desejado
    embedding_resumo = model.encode(resumo, convert_to_tensor=True)
    sim_manter_r = util.cos_sim(embedding_resumo, embedding_manter_r).item()
    sim_eliminar_r = util.cos_sim(embedding_resumo, embedding_eliminar_r).item()

    # Classificação com a margem restrita de corte calibrada em >= 0.55 para os resumos
    if sim_manter_r > sim_eliminar_r and sim_manter_r >= 0.55:
        df_resumos.at[idx, 'Decisao_Resumo'] = "MANTER"
        df_resumos.at[idx, 'Score_Resumo'] = round(sim_manter_r, 4)
    else:
        df_resumos.at[idx, 'Decisao_Resumo'] = "ELIMINAR"
        df_resumos.at[idx, 'Score_Resumo'] = round(sim_eliminar_r, 4)

output_file_r = "banco_resumos_filtrados.xlsx"
df_resumos.to_excel(output_file_r, index=False)
print(f"\nFiltragem finalizada com sucesso! Baixando portfólio purificado: {output_file_r}")
files.download(output_file_r)